# Lab Day 10: Finetune Qwen3-VL-4B (LoRA) để định vị hư hại mặt đường — bản chạy trên Kaggle

Notebook này chỉ **điều phối**: mã thật nằm trong package `d10/` của repo (mỗi thí nghiệm chạy thành một tiến trình riêng để GPU được giải phóng hoàn toàn giữa các lần chạy). Công cụ của đề trong `scripts/` và `tests/` không bị sửa.

**Trước khi chạy**

1. Push repo lên GitHub (repo công khai), rồi sửa `REPO_URL` ở ô đầu nếu cần.
2. Kaggle → *Settings*: **Accelerator = GPU T4 x2**, **Internet = On**.
3. Dữ liệu: không cần làm gì, notebook tự tải zip công khai của dataset `lorenzoarcioni/road-damage-dataset-potholes-cracks-and-manholes` (hoặc dùng bản đã gắn qua *Add Input* nếu có).
4. (Tuỳ chọn) *Add-ons → Secrets*: `HF_TOKEN` để tải model nhanh hơn. Không dán khoá vào notebook.

**Cách chạy:** *Save Version → Save & Run All (Commit)* để chạy nền, hoặc *Run All* trong phiên tương tác. Mọi bước đều bỏ qua phần đã có kết quả, nên chạy lại sau khi lỗi sẽ tiếp tục từ chỗ dở (trong cùng một phiên).

**Kết quả:** ô cuối đóng gói `report/` (evidence, log, ảnh minh hoạ, output thô) thành `/kaggle/working/day10_results.zip`. Tải về, giải nén vào thư mục gốc của repo rồi commit.

## 0. Môi trường (Kaggle T4)

In [ ]:
REPO_URL = "https://github.com/DAQuan-VinAI/K4-Track-4-Day10-VisionLanguageModel.git"
BRANCH = "main"
RUN_SMOKE = True   # chạy thử toàn pipeline 2 bước / 8 ảnh trước khi chạy thật
RUN_E5 = True      # bonus GRPO (tuỳ chọn)

import os
ROOT = "/kaggle/working/lab"
if not os.path.isdir(f"{ROOT}/.git"):
    !git clone -q -b $BRANCH $REPO_URL $ROOT
%cd $ROOT
!git pull -q --ff-only
!git log --oneline -1
assert os.path.isdir("d10") and os.path.isdir("tests_d10"), (
    "Repo trên GitHub chưa có thư mục d10/: commit + push code ở máy lên nhánh " + BRANCH + " rồi chạy lại")
!nvidia-smi -L
!pip -q install -r requirements-colab.txt

In [ ]:
# HF_TOKEN là tuỳ chọn và chỉ đọc từ Kaggle Secrets. KHÔNG viết khoá vào notebook.
try:
    from kaggle_secrets import UserSecretsClient
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
    print("HF_TOKEN: lấy từ Kaggle Secrets")
except Exception:
    print("Không có secret HF_TOKEN: tải model ẩn danh")

## 1. Dữ liệu: tải và chia split

In [ ]:
import glob
SLUG = "lorenzoarcioni/road-damage-dataset-potholes-cracks-and-manholes"
hits = glob.glob("/kaggle/input/**/labels-YOLO", recursive=True)
if hits:                                   # dataset đã gắn qua Add Input
    SRC = os.path.dirname(hits[0])
    print("nguồn dữ liệu:", SRC)
    !python scripts/prepare_data.py --source "$SRC"
else:
    # Dataset công khai (MIT): tải zip thẳng từ API của Kaggle, không cần khoá và không cần Add Input.
    ZIP_URL = f"https://www.kaggle.com/api/v1/datasets/download/{SLUG}"
    print("nguồn dữ liệu:", ZIP_URL)
    !python scripts/prepare_data.py --zip "$ZIP_URL"
assert _exit_code == 0, "prepare_data.py lỗi"

In [ ]:
# Công cụ của đề + test của phần mã tự viết (nhãn, reward)
!python -m pytest -q
!python -m pytest -q tests_d10

In [ ]:
import json
from collections import Counter
from scripts.labcommon import read_jsonl

splits = {s: read_jsonl(f"data/splits/{s}.jsonl") for s in ("train", "val", "test")}
for s, rows in splits.items():
    c = Counter(o["label"] for r in rows for o in r["objects"])
    print(f"{s:5s} {len(rows):5d} ảnh  {sum(c.values()):5d} box  {dict(c)}  "
          f"ảnh không có box: {sum(not r['objects'] for r in rows)}")

# Không phiên quay nào được nằm ở hai split (chống rò rỉ giữa các frame liền kề)
sess = {s: {r["session"] for r in rows} for s, rows in splits.items()}
assert not (sess["train"] & sess["val"] or sess["train"] & sess["test"] or sess["val"] & sess["test"])
print({s: len(v) for s, v in sess.items()}, "phiên; không phiên nào trùng giữa các split")
print("dataset_version:", json.load(open("data/splits/meta.json"))["dataset_version"])

In [ ]:
# 6 ảnh train có box. Box vẽ từ nhãn SAU khi đổi sang thang 0-1000 (bước 2),
# nên nếu phép đổi sai thì box sẽ lệch khỏi vật thể ngay ở đây.
import random
import matplotlib.pyplot as plt
from PIL import Image
from d10.viz import draw, gt_as_seen_by_model

sample = random.Random(0).sample([r for r in splits["train"] if r["objects"]], 6)
fig, axes = plt.subplots(2, 3, figsize=(18, 7))
for ax, r in zip(axes.flat, sample):
    ax.imshow(draw(Image.open(f"data/{r['image_path']}"), gt=gt_as_seen_by_model(r)))
    ax.set_title(f"{r['image_id']} · {r['session']}", fontsize=9)
    ax.axis("off")
plt.tight_layout()
plt.show()

## 2. Nhãn: YOLO → JSON thang 0–1000

In [ ]:
import inspect
from d10 import labels

print(inspect.getsource(labels.yolo_to_box))
print(inspect.getsource(labels.objects_to_answer))
print("PROMPT:", labels.PROMPT)

In [ ]:
# Kiểm tra: đổi nhãn sang JSON 0-1000 rồi quy ngược về pixel phải trùng box gốc
# (sai số chỉ do làm tròn: 0,5/1000 cạnh ảnh, tức < 0,4 px với ảnh 640x360).
worst = 0.0
for r in splits["train"]:
    w, h = r["width"], r["height"]
    for o in r["objects"]:
        xc, yc, bw, bh = o["yolo"]
        orig = ((xc - bw / 2) * w, (yc - bh / 2) * h, (xc + bw / 2) * w, (yc + bh / 2) * h)
        back = labels.box_to_pixels(labels.yolo_to_box(o["label"], o["yolo"])["bbox_2d"], w, h)
        worst = max(worst, *(abs(a - b) for a, b in zip(orig, back)))
print(f"sai số lớn nhất sau khi quy ngược: {worst:.3f} px")
assert worst < 1.0

r = sample[0]
print("ví dụ đáp án:", labels.objects_to_answer(r["objects"]))

# Mỗi box tốn khoảng 29 token, nên max_new_tokens quyết định số box tối đa trả về được.
for s, rows in splits.items():
    n = sorted(len(r["objects"]) for r in rows)
    print(f"{s:5s} box/ảnh: trung vị {n[len(n) // 2]}, p99 {n[int(0.99 * len(n))]}, max {n[-1]}, "
          f"ảnh > 8 box: {sum(x > 8 for x in n)}")

## 3. E0: zero-shot

In [ ]:
# Chạy thử toàn bộ pipeline (2 bước huấn luyện, 8 ảnh) để bắt lỗi môi trường trước khi chạy thật.
# Kết quả thử nằm ở work/smoke/, KHÔNG ghi vào report/evidence.
if RUN_SMOKE:
    !python -m d10.pipeline e0 e1 e3 --smoke
    assert _exit_code == 0, "Chạy thử lỗi: xem log phía trên trước khi tốn giờ GPU"

In [ ]:
# E0: model gốc 4-bit, suy luận trên test -> work/e0.jsonl -> scripts/score.py -> report/evidence/e0.json
!python -m d10.pipeline e0
assert _exit_code == 0, "E0 lỗi"

In [ ]:
# 5 output thô của E0: có bọc ```json không, có chữ thừa không, toạ độ theo thang nào?
for row in read_jsonl("work/e0.jsonl")[:5]:
    print(row["image_id"], "->", repr(row["raw"][:300]))

## 4. Dựng dữ liệu huấn luyện: chat template và mask loss

In [ ]:
from d10 import data
print(inspect.getsource(data.encode_example))
print(inspect.getsource(data.collate))

In [ ]:
# Kiểm tra trên một ví dụ: 220 token ảnh cho ảnh 640x360, và phần có labels != -100
# phải đúng là JSON đáp án (kèm <|im_end|>).
from d10.modeling import load_config, load_processor

processor = load_processor(load_config("configs/e1.yaml")["model_id"])
r = sample[0]
answer = labels.objects_to_answer(r["objects"])
ex = data.encode_example(processor, Image.open(f"data/{r['image_path']}").convert("RGB"), answer)
n_img = int((ex["input_ids"] == processor.image_token_id).sum())
kept = ex["labels"][ex["labels"] != data.IGNORE]
print("số token ảnh      :", n_img)
print("tổng số token     :", ex["input_ids"].shape[1], "| token tính loss:", len(kept))
print("giải mã phần nhãn :", repr(processor.tokenizer.decode(kept)))
assert n_img == 220
assert processor.tokenizer.decode(kept).startswith(answer + "<|im_end|>")

## 5. Setup LoRA / đóng băng cho E1, E2, E3

In [ ]:
# Số module khớp regex (252 cho E1/E2, 348 cho E3) được kiểm tra bằng raise ngay trong apply_setup
# và in ra ở đầu log huấn luyện của từng thí nghiệm (bước 6), cùng số tham số huấn luyện.
from d10 import modeling
print("LLM:", modeling.LLM_TARGETS)
print("ViT:", modeling.VIT_TARGETS)
print(inspect.getsource(modeling.apply_setup))
print(inspect.getsource(modeling.save_trained))

## 6. Vòng huấn luyện

In [ ]:
from d10 import train
print(inspect.getsource(train.lr_factor))
print(inspect.getsource(train.train))

In [ ]:
# E1, E2, E3: mỗi cái 300 bước tối ưu (batch 2 x tích luỹ 4). Adapter + projector lưu ở work/runs/eN/.
!python -m d10.pipeline e1 e2 e3 --stage train
assert _exit_code == 0, "Huấn luyện lỗi (hết VRAM? xem bảng 'Khi gặp sự cố' trong GUIDE)"

## 7. Suy luận và chấm E1–E3

In [ ]:
# Nạp model gốc 4-bit -> nạp lại projector đã huấn luyện (E2/E3) -> gắn adapter -> suy luận test -> score.py
!python -m d10.pipeline e1 e2 e3 --stage eval
assert _exit_code == 0, "Suy luận/chấm lỗi"

In [ ]:
!python -m d10.summary

In [ ]:
# Bẫy thang toạ độ (câu 3 của báo cáo): cùng một file dự đoán của E1 (thang 0-1000),
# nhưng chấm như thể đó là pixel. Kết quả ghi vào work/, không phải evidence.
!python scripts/score.py --pred work/e1.jsonl --name E1_cham_nham_pixel --coord pixel --out work/trap/e1_as_pixel.json

In [ ]:
# Ảnh minh hoạ: 4 ảnh đúng hết + 4 ảnh sai nhiều nhất cho E0 và E1 (xanh = nhãn đúng, đỏ = dự đoán)
from IPython.display import Image as ShowImage, display

!python -m d10.viz --pred work/e0.jsonl --name E0
!python -m d10.viz --pred work/e1.jsonl --name E1 --loss
for f in sorted(glob.glob("report/figures/E1_*.jpg")):
    display(ShowImage(f))
display(ShowImage("report/figures/loss.png"))

## 8. E4 (ablation tự chọn) và E5 (tuỳ chọn, GRPO)

In [ ]:
# E4 = data_size: lấy setup F1 cao nhất trong E1-E3, CHỈ đổi số ảnh train (10% so với 100%).
# 100% chính là lần chạy gốc, nên chỉ cần huấn luyện thêm một lần.
!python -m d10.pipeline e4
assert _exit_code == 0, "E4 lỗi"

In [ ]:
# E5 (bonus): GRPO với reward IoU, khởi tạo từ adapter tốt nhất. Lỗi ở đây không chặn phần còn lại.
if RUN_E5:
    print(inspect.getsource(__import__("d10.reward", fromlist=["reward"]).reward))
    !python -m d10.pipeline e5
    print("E5 exit code:", _exit_code)

In [ ]:
!python -m d10.summary
!python -m d10.viz --loss
if os.path.exists("work/e5.jsonl"):
    !python -m d10.viz --pred work/e5.jsonl --name E5
# REPORT.md chưa điền và số commit sẽ còn FAIL ở đây; các mục evidence phải PASS.
!python scripts/check_submission.py

In [ ]:
# Đóng gói kết quả để tải về: report/ (evidence, logs, predictions, figures, summary.md)
# và data/splits/meta.json (chỉ để check_submission.py đối chiếu ở máy; data/ không được commit).
import zipfile
from pathlib import Path

OUT = "/kaggle/working/day10_results.zip"
with zipfile.ZipFile(OUT, "w", zipfile.ZIP_DEFLATED) as z:
    for p in sorted(Path("report").rglob("*")):
        if p.is_file():
            z.write(p)
    z.write("data/splits/meta.json")
print(OUT, f"{os.path.getsize(OUT) / 1e6:.1f} MB")
!ls -la report/evidence